In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os 
from langchain_prompty import create_chat_prompt
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain

ModuleNotFoundError: No module named 'langchain.chains'

In [13]:
loader=PyPDFLoader("Tushar_chaudhari_26.pdf")
docs = loader.load()
docs

[Document(metadata={'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-05-15T12:34:51+00:00', 'author': '', 'keywords': '', 'moddate': '2026-05-15T12:34:51+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'Tushar_chaudhari_26.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='Tushar Shivaji Chaudhari\nDhule, Maharashtra|+91 9607702282|tc3158494@gmail.com|LinkedIn|GitHub\nEDUCATION\nZ.B. Patil College, Dhule, NMU 2023 – 2026\nB.Sc. Computer Science\nPROJECTS\nGenAI Project: Y ouT ube Video & W ebsite URL Content Summarization App|Python, Streamlit,\nLangChain, HuggingF ace\n–Built a Generative AI application that summarizes YouTube videos and website content into concise, human-readable\ninsights using advanced LLM workflows.\n–Integrated AI-driven content extraction and summarization pipelines to help users q

In [17]:
text_splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50)
final_documents = text_splitter.split_documents(docs)

In [22]:
embeddings=OllamaEmbeddings(model="nomic-embed-text")
vectordb=Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    
)

In [32]:
load_dotenv()

api_key= os.environ["GROQ_API_KEY"]

In [33]:
llm=ChatGroq(model="llama-3.1-8b-instant")

In [41]:

prompt = create_chat_prompt("""
You are an intelligent AI assistant specialized in answering questions from uploaded PDF documents.

Instructions:
- Use ONLY the provided PDF context to answer the question.
- If the answer is not found in the context, say:
  "The answer is not available in the uploaded PDF document."

PDF Context:
{context}

User Question:
{question}

Answer:
""")

In [44]:
retriever = vectordb.as_retriever()

In [ ]:
document_chain = create_stuff_documents_chain(
    llm,
    prompt
)

In [ ]:
retrieval_chain = create_retrieval_chain(
    retriever,
    document_chain
)


[Document(metadata={'page_label': '1', 'title': '', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'total_pages': 1, 'creationdate': '2026-05-15T12:34:51+00:00', 'author': '', 'producer': 'pdfTeX-1.40.27', 'page': 0, 'subject': '', 'trapped': '/False', 'source': 'Tushar_chaudhari_26.pdf', 'moddate': '2026-05-15T12:34:51+00:00', 'keywords': '', 'creator': 'LaTeX with hyperref'}, page_content='Tushar Shivaji Chaudhari\nDhule, Maharashtra|+91 9607702282|tc3158494@gmail.com|LinkedIn|GitHub\nEDUCATION\nZ.B. Patil College, Dhule, NMU 2023 – 2026\nB.Sc. Computer Science\nPROJECTS\nGenAI Project: Y ouT ube Video & W ebsite URL Content Summarization App|Python, Streamlit,\nLangChain, HuggingF ace\n–Built a Generative AI application that summarizes YouTube videos and website content into concise, human-readable\ninsights using advanced LLM workflows.\n–Integrated AI-driven content extraction and summarization pipelines to help users q

In [ ]:
response = retrieval_chain.invoke({
    "input": "What is this PDF about?"
})